## Imports & Display Settings

In [1]:
# Standard imports
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning & modelling
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Pandas display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.expand_frame_repr", False)

# Seaborn style
sns.set(style="whitegrid", font_scale=1.2)

## Load data

In [2]:
# Matches
matches = pd.read_parquet(r"Datasets/SkillCorner Premier League 24-25 data/matches_clean.parquet")

# Team lookup table
team_lookup = pd.concat([
    matches[["home_team_id","home_team_name"]].rename(columns={"home_team_id":"team_id","home_team_name":"team_name"}),
    matches[["away_team_id","away_team_name"]].rename(columns={"away_team_id":"team_id","away_team_name":"team_name"})
]).drop_duplicates()

# Event files
folder = Path(r"Datasets/SkillCorner Premier League 24-25 data\dynamic_events_pl_24\dynamic")
dfs = [pd.read_parquet(file) for file in folder.glob("*.parquet")]
events = pd.concat(dfs, ignore_index=True)

print(f"Total events: {len(events)}, Unique matches: {events['match_id'].nunique()}")

Total events: 1811078, Unique matches: 378


## Check player_targeted_xpass_completion

In [3]:
# Check missing values per event type
for event_type in ["player_possession", "passing_option"]:
    ev = events[events["event_type"] == event_type]
    total = len(ev)
    non_missing = ev["player_targeted_xpass_completion"].notna().sum()
    print(f"{event_type}: {non_missing}/{total} non-missing ({non_missing/total:.2%})")

player_possession: 306381/362853 non-missing (84.44%)
passing_option: 0/939059 non-missing (0.00%)


## Preprocessing and Feature Engineering

In [4]:
# 1️⃣ Filter passing options
pass_cols = [
    "match_id",
    "associated_player_possession_event_id",
    "player_id",        # receiver
    "player_name",
    "xthreat",
    "targeted",
    "x_start",          # receiver x
    "y_start",          # receiver y
    "interplayer_angle",
    "interplayer_distance",
    "interplayer_direction",
    "channel_start",
    "third_start",
    "penalty_area_start",
    "organised_defense"
]

passing_options = events[events["event_type"] == "passing_option"][pass_cols].copy()

# Rename receiver coordinates
passing_options = passing_options.rename(columns={"x_start": "receiver_x", "y_start": "receiver_y"})

# Convert numeric features
passing_options["xthreat"] = pd.to_numeric(passing_options["xthreat"], errors="coerce").fillna(0)

# 2️⃣ Merge possession info
possession_cols = [
    "match_id", "event_id", "player_id", "player_name",
    "x_start", "y_start", "player_targeted_id", "player_targeted_xpass_completion"
]
possession_events = events[events["event_type"] == "player_possession"][possession_cols].copy()
possession_events = possession_events.rename(columns={
    "event_id": "possession_event_id",
    "player_id": "passer_id",
    "player_name": "passer_name",
    "x_start": "passer_x",
    "y_start": "passer_y",
    "player_targeted_id": "target_receiver_id",
    "player_targeted_xpass_completion": "xpass_from_model"
})

passing_options = passing_options.merge(
    possession_events,
    left_on=["match_id", "associated_player_possession_event_id"],
    right_on=["match_id", "possession_event_id"],
    how="left"
)

# 3️⃣ Assign xpass_completion for targeted receiver
mask = passing_options["player_id"] == passing_options["target_receiver_id"]
passing_options["xpass_completion"] = None
passing_options.loc[mask, "xpass_completion"] = passing_options.loc[mask, "xpass_from_model"]

# 4️⃣ Feature engineering
passing_options["interplayer_angle_sin"] = np.sin(passing_options["interplayer_angle"])
passing_options["interplayer_angle_cos"] = np.cos(passing_options["interplayer_angle"])
passing_options["passer_receiver_x_diff"] = passing_options["receiver_x"] - passing_options["passer_x"]
passing_options["passer_receiver_y_diff"] = passing_options["receiver_y"] - passing_options["passer_y"]

# Drop unused columns
passing_options = passing_options.drop(columns=["xpass_from_model", "possession_event_id", "target_receiver_id"])

## Define Feature Groups

In [5]:
# Tabular / categorical / additional / spatial
tabular_features = ["xthreat", "interplayer_distance"]
categorical_features = ["interplayer_direction", "channel_start", "third_start", "penalty_area_start", "organised_defense"]
additional_features = ["interplayer_angle_sin", "interplayer_angle_cos", "passer_receiver_x_diff", "passer_receiver_y_diff"]
spatial_features = ["receiver_x", "receiver_y", "passer_x", "passer_y"]
features_all = tabular_features + categorical_features + additional_features

## Split Dataset for Model Training

In [6]:
target = "xpass_completion"
train_df = passing_options[passing_options[target].notna()].copy()
predict_df = passing_options[passing_options[target].isna()].copy()
train_df[target] = train_df[target].astype(float)

# One-hot encode categorical features
X = pd.get_dummies(train_df[features_all])
y = train_df[target]
X_pred = pd.get_dummies(predict_df[features_all]).reindex(columns=X.columns, fill_value=0)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=14)

## Train Tree Models

In [7]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=200, max_depth=12, n_jobs=-1, random_state=14),
    "Hist Gradient Boosting": HistGradientBoostingRegressor(max_iter=400, learning_rate=0.05, max_depth=6, random_state=14),
    "XGBoost": xgb.XGBRegressor(n_estimators=500, learning_rate=0.03, max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=14, n_jobs=-1),
    "LightGBM": lgb.LGBMRegressor(n_estimators=500, learning_rate=0.03, max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=14, n_jobs=-1),
    "CatBoost": cb.CatBoostRegressor(iterations=500, learning_rate=0.03, depth=6, verbose=0, random_state=14)
}

results = []
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    preds = np.clip(model.predict(X_test), 0, 1)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    results.append((name, rmse, mae))
    print(f"{name} evaluation: RMSE={rmse:.4f}, MAE={mae:.4f}\n{'-'*40}")

results_df = pd.DataFrame(results, columns=["Model", "RMSE", "MAE"])
best_model_name = results_df.sort_values("RMSE").iloc[0]["Model"]
best_model = models[best_model_name]
print("Best tree model:", best_model_name)

# Predict missing xpass_completion
predict_df["predicted_xpass"] = np.clip(best_model.predict(X_pred), 0, 1)
passing_options.loc[predict_df.index, "predicted_xpass"] = predict_df["predicted_xpass"]

Training Linear Regression...
Linear Regression evaluation: RMSE=0.1387, MAE=0.0977
----------------------------------------
Training Random Forest...
Random Forest evaluation: RMSE=0.1071, MAE=0.0668
----------------------------------------
Training Hist Gradient Boosting...
Hist Gradient Boosting evaluation: RMSE=0.1040, MAE=0.0649
----------------------------------------
Training XGBoost...
XGBoost evaluation: RMSE=0.1036, MAE=0.0645
----------------------------------------
Training LightGBM...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002190 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1560
[LightGBM] [Info] Number of data points in the train set: 282711, number of used features: 21
[LightGBM] [Info] Start training from score 0.865602
LightGBM evaluation: RMSE=0.1042, MAE=0.0651
----------------------------------------
Trai

## Neural Networks

In [8]:
# Tensors
X_tab_t = torch.tensor(X_train.values, dtype=torch.float32)
X_sp_t = torch.tensor(train_df[spatial_features].values, dtype=torch.float32)
y_t = torch.tensor(y_train.values.reshape(-1,1), dtype=torch.float32)

X_tab_tr, X_tab_val, X_sp_tr, X_sp_val, y_tr, y_val = train_test_split(
    X_tab_t, X_sp_t, y_t, test_size=0.1, random_state=14
)

train_loader = DataLoader(TensorDataset(X_tab_tr, X_sp_tr, y_tr), batch_size=2048, shuffle=True)
val_loader = DataLoader(TensorDataset(X_tab_val, X_sp_val, y_val), batch_size=2048, shuffle=False)

class SpatialPassNN(nn.Module):
    def __init__(self, tab_dim, spatial_dim, tab_hidden=128, spatial_hidden=64, dropout=0.3):
        super().__init__()
        self.tab_branch = nn.Sequential(
            nn.Linear(tab_dim, tab_hidden), nn.ReLU(),
            nn.BatchNorm1d(tab_hidden), nn.Dropout(dropout),
            nn.Linear(tab_hidden, tab_hidden//2), nn.ReLU()
        )
        self.spatial_branch = nn.Sequential(
            nn.Linear(spatial_dim, spatial_hidden), nn.ReLU(),
            nn.BatchNorm1d(spatial_hidden), nn.Dropout(dropout),
            nn.Linear(spatial_hidden, spatial_hidden//2), nn.ReLU()
        )
        self.combined = nn.Sequential(
            nn.Linear(tab_hidden//2 + spatial_hidden//2, 64), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(64,1), nn.Sigmoid()
        )
    def forward(self, x_tab, x_spatial):
        t = self.tab_branch(x_tab)
        s = self.spatial_branch(x_spatial)
        return self.combined(torch.cat([t,s], dim=1))

TypeError: can't convert np.ndarray of type numpy.object_. The only supported types are: float64, float32, float16, complex64, complex128, int64, int32, int16, int8, uint64, uint32, uint16, uint8, and bool.

In [9]:
# Ensure all categorical / additional features are one-hot encoded
X_tab_full = pd.get_dummies(train_df[features_all])  # all tabular + engineered features
X_pred_full = pd.get_dummies(predict_df[features_all]).reindex(columns=X_tab_full.columns, fill_value=0)

# Ensure numeric type
X_tab_full = X_tab_full.astype(np.float32)
X_pred_full = X_pred_full.astype(np.float32)
y_t = y_train.astype(np.float32).values.reshape(-1, 1)

# Split tabular and spatial features
X_tab_t = torch.tensor(X_tab_full.values, dtype=torch.float32)
X_sp_t = torch.tensor(train_df[spatial_features].values.astype(np.float32), dtype=torch.float32)
y_t = torch.tensor(y_t, dtype=torch.float32)

# Train/validation split
X_tab_tr, X_tab_val, X_sp_tr, X_sp_val, y_tr, y_val = train_test_split(
    X_tab_t, X_sp_t, y_t, test_size=0.1, random_state=14
)

# DataLoaders
train_loader = DataLoader(TensorDataset(X_tab_tr, X_sp_tr, y_tr), batch_size=2048, shuffle=True)
val_loader = DataLoader(TensorDataset(X_tab_val, X_sp_val, y_val), batch_size=2048, shuffle=False)

ValueError: Found input variables with inconsistent numbers of samples: [314124, 314124, 282711]

## Neural Network Training & Hyperparameter Search

In [ ]:
param_grid = {"tab_hidden":[64,128],"spatial_hidden":[32,64],"dropout":[0.2,0.3],"lr":[0.001,0.0005]}
best_val_rmse = float('inf')
best_params = None
best_model_state = None

for params in ParameterGrid(param_grid):
    model = SpatialPassNN(tab_dim=X_tab_t.shape[1], spatial_dim=X_sp_t.shape[1],
                          tab_hidden=params["tab_hidden"], spatial_hidden=params["spatial_hidden"],
                          dropout=params["dropout"])
    optimizer = optim.Adam(model.parameters(), lr=params["lr"])
    criterion = nn.MSELoss()
    patience = 5
    patience_counter = 0
    for epoch in range(20):
        model.train()
        for xb_tab, xb_sp, yb in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(xb_tab, xb_sp), yb)
            loss.backward()
            optimizer.step()
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for xb_tab, xb_sp, yb in val_loader:
                val_loss += criterion(model(xb_tab, xb_sp), yb).item() * xb_tab.size(0)
        val_rmse = np.sqrt(val_loss / len(val_loader.dataset))
        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_params = params
            best_model_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

print("Best NN params:", best_params, "Val RMSE:", best_val_rmse)

## Neural Network Prediction

In [ ]:
best_nn = SpatialPassNN(tab_dim=X_tab_t.shape[1], spatial_dim=X_sp_t.shape[1],
                        tab_hidden=best_params["tab_hidden"], spatial_hidden=best_params["spatial_hidden"],
                        dropout=best_params["dropout"])
best_nn.load_state_dict(best_model_state)
best_nn.eval()

X_pred_tab_t = torch.tensor(X_pred.values, dtype=torch.float32)
X_pred_sp_t = torch.tensor(predict_df[spatial_features].values, dtype=torch.float32)

with torch.no_grad():
    preds_nn = best_nn(X_pred_tab_t, X_pred_sp_t).numpy().flatten()

predict_df["predicted_xpass_nn"] = np.clip(preds_nn,0,1)
passing_options.loc[predict_df.index, "predicted_xpass_nn"] = predict_df["predicted_xpass_nn"]

## Model Evaluation on Known xpass_completion

In [ ]:
X_test_full = pd.get_dummies(train_df[tabular_features + categorical_features + additional_features])
X_test_full = X_test_full.reindex(columns=X.columns, fill_value=0)
y_true = train_df["xpass_completion"]

# Tree prediction
tree_preds = best_model.predict(X_test_full)
tree_rmse = np.sqrt(mean_squared_error(y_true, tree_preds))
tree_mae = mean_absolute_error(y_true, tree_preds)

# NN prediction
X_tab_t_full = torch.tensor(X_test_full.values, dtype=torch.float32)
X_sp_t_full = torch.tensor(train_df[spatial_features].values, dtype=torch.float32)
with torch.no_grad():
    nn_preds = best_nn(X_tab_t_full, X_sp_t_full).numpy().flatten()
nn_rmse = np.sqrt(mean_squared_error(y_true, nn_preds))
nn_mae = mean_absolute_error(y_true, nn_preds)

print(f"Model comparison on known xpass_completion:")
print(f"{best_model_name} -> RMSE: {tree_rmse:.4f}, MAE: {tree_mae:.4f}")
print(f"Neural Network -> RMSE: {nn_rmse:.4f}, MAE: {nn_mae:.4f}")

## Final Prediction for Missing Values

In [ ]:
# Tree model prediction
preds = np.clip(best_model.predict(X_pred), 0, 1)
predict_df["predicted_xpass"] = preds
passing_options["predicted_xpass"] = None
passing_options.loc[predict_df.index, "predicted_xpass"] = preds

print("Predictions added. Range:", passing_options["predicted_xpass"].min(), passing_options["predicted_xpass"].max())

# Final analysis-ready dataset
passing_options.head()

## EVALUATION OF MODELS ON KNOWN xpass_completion

In [ ]:
X_test_full = pd.get_dummies(train_df[tabular_features + categorical_features + additional_features])
X_test_full = X_test_full.reindex(columns=X.columns, fill_value=0)
y_true = train_df["xpass_completion"]

# Tree prediction
tree_preds = best_model.predict(X_test_full)
tree_rmse = np.sqrt(mean_squared_error(y_true,tree_preds))
tree_mae = mean_absolute_error(y_true,tree_preds)

# NN prediction
X_tab_t_full = torch.tensor(X_test_full.values,dtype=torch.float32)
X_sp_t_full = torch.tensor(train_df[spatial_features].values,dtype=torch.float32)
with torch.no_grad():
    nn_preds = best_nn(X_tab_t_full,X_sp_t_full).numpy().flatten()
nn_rmse = np.sqrt(mean_squared_error(y_true,nn_preds))
nn_mae = mean_absolute_error(y_true,nn_preds)

print(f"Model comparison on known xpass_completion:")
print(f"{best_model_name} -> RMSE: {tree_rmse:.4f}, MAE: {tree_mae:.4f}")
print(f"Neural Network -> RMSE: {nn_rmse:.4f}, MAE: {nn_mae:.4f}")

## Predict xpass_completion for Missing Values

In [ ]:
# Predict using the best tree model (XGBoost)
preds = np.clip(best_model.predict(X_pred), 0, 1)
predict_df["predicted_xpass"] = preds
passing_options["predicted_xpass"] = None
passing_options.loc[predict_df.index, "predicted_xpass"] = preds

print("Predictions added. Range:", passing_options["predicted_xpass"].min(), passing_options["predicted_xpass"].max())

## Analysis-Ready Dataset

In [ ]:
# Now you have a dataset with:
# - all passing options
# - actual xpass_completion for targeted passes
# - predicted xpass_completion for missing ones
passing_options.head()

## Work with xpass_completion

In [ ]:
# Count total and missing values
total_values = len(events)
missing_values = events["player_targeted_xpass_completion"].isna().sum()
actual_values = total_values - missing_values

print(f"Total rows: {total_values}")
print(f"Missing values: {missing_values} ({missing_values/total_values:.2%})")
print(f"Actual values: {actual_values} ({actual_values/total_values:.2%})")

# Unique event types
event_types = events["event_type"].unique()
print("Unique event types in dataset:")
print(event_types)

In [ ]:
# List of event types to analyse
event_types = ["player_possession", "passing_option"]

for event in event_types:
    # Filter rows for the event
    event_rows = events[events["event_type"] == event]
    
    # Filter rows where xpass_completion exists
    xpass_rows = event_rows[event_rows["player_targeted_xpass_completion"].notna()]
    
    total_rows = len(event_rows)
    xpass_count = len(xpass_rows)
    percentage = (xpass_count / total_rows * 100) if total_rows > 0 else 0
    
    print(f"\nEvent type: {event}")
    print(f"Total rows: {total_rows}")
    print(f"Rows with player_targeted_xpass_completion: {xpass_count} ({percentage:.2f}%)")
    
    # If 'targeted' column exists, show distribution
    if "targeted" in xpass_rows.columns:
        targeted_dist = xpass_rows.groupby("targeted").size().reset_index(name="count")
        targeted_dist["percentage"] = targeted_dist["count"] / xpass_count * 100
        print("\nDistribution of 'targeted' column:")
        print(targeted_dist)

In [ ]:
# Filter only passing_option events
passing_options = events[events["event_type"] == "passing_option"].copy()

# Convert to numeric, just in case
passing_options["player_targeted_xpass_completion"] = pd.to_numeric(
    passing_options["player_targeted_xpass_completion"], errors="coerce"
)

# Count missing and non-missing
total_rows = len(passing_options)
missing = passing_options["player_targeted_xpass_completion"].isna().sum()
non_missing = total_rows - missing
print(f"Total passing_option rows: {total_rows}")
print(f"Missing values: {missing} ({missing/total_rows:.2%})")
print(f"Non-missing values: {non_missing} ({non_missing/total_rows:.2%})")

# Basic statistics
print("\nStatistics for player_targeted_xpass_completion (non-missing values):")
print(passing_options["player_targeted_xpass_completion"].describe())

# Check relationship with 'targeted'
print("\nDistribution of targeted for rows with non-missing xpass_completion:")
print(passing_options.loc[passing_options["player_targeted_xpass_completion"].notna(), "targeted"].value_counts())

In [ ]:
example_row = passing_options.iloc[100]

example_event = example_row["associated_player_possession_event_id"]
example_match = example_row["match_id"]

print("Match:", example_match)
print("Possession event:", example_event)

In [ ]:
possession_event = events[
    (events["event_type"] == "player_possession") &
    (events["event_id"] == example_event) &
    (events["match_id"] == example_match)
]

possession_event = possession_event.sort_values("frame_start").iloc[-1]

possession_event.to_frame().T

In [ ]:
pass_columns = [
    "event_id", "associated_player_possession_event_id", "match_id", "time_start", "period", "team_shortname",
    "player_id",
    "player_name",
    "xthreat",
    "player_targeted_id",
    "player_targeted_name",
    "player_targeted_xpass_completion",
    "player_targeted_xthreat",

    "targeted",

    "x_start",
    "y_start",
    "channel_start", "third_start", "penalty_area_start", 
    "game_state", "phase_index", "team_in_possession_phase_type_id", "team_in_possession_phase_type", "team_out_of_possession_phase_type",

    "player_in_possession_x_start",
    "player_in_possession_y_start",
    "speed_avg",
    "end_type",
    "one_touch", "quick_pass", "is_header", "hand_pass", "forward_momentum",

    "pass_distance",
    "pass_angle",
    "pass_direction",
    "pass_range",
    "pass_outcome",

    "n_passing_options",
    "n_passing_options_ahead",
    "n_opponents_ahead_player_in_possession_pass_moment",
    "n_opponents_ahead_pass_reception",
    "n_opponents_bypassed",

    "interplayer_distance",
    "interplayer_angle",
    "interplayer_direction",

    "organised_defense"
]

In [ ]:
possession_event_filtered = possession_event[pass_columns]
possession_event_filtered.to_frame().T

In [ ]:
options = events[
    (events["event_type"] == "passing_option") &
    (events["associated_player_possession_event_id"] == example_event) &
    (events["match_id"] == example_match)
]

options = options[pass_columns]

options

In [ ]:
# ==================================================
# RELEVANT COLUMNS
# ==================================================

pass_columns = [
    "event_id", "associated_player_possession_event_id", "match_id", "time_start", "period", "team_shortname",
    "player_id", "player_name",
    "xthreat",
    "player_targeted_id", "player_targeted_name",
    "player_targeted_xpass_completion", "player_targeted_xthreat",
    "targeted",
    "x_start", "y_start",
    "channel_start", "third_start", "penalty_area_start",
    "game_state", "phase_index",
    "team_in_possession_phase_type_id", "team_in_possession_phase_type",
    "team_out_of_possession_phase_type",
    "player_in_possession_x_start", "player_in_possession_y_start",
    "speed_avg",
    "end_type",
    "one_touch", "quick_pass", "is_header", "hand_pass", "forward_momentum",
    "pass_distance", "pass_angle", "pass_direction", "pass_range", "pass_outcome",
    "n_passing_options", "n_passing_options_ahead",
    "n_opponents_ahead_player_in_possession_pass_moment",
    "n_opponents_ahead_pass_reception",
    "n_opponents_bypassed",
    "interplayer_distance", "interplayer_angle", "interplayer_direction",
    "organised_defense"
]

In [ ]:
# ==================================================
# 1️⃣ FILTER PASSING OPTIONS
# ==================================================

passing_options = events[events["event_type"] == "passing_option"].copy()
passing_options = passing_options[pass_columns].copy()

# Remove empty column if present
passing_options = passing_options.drop(columns=["pass_outcome"], errors="ignore")

# Rename receiver position (passing option)
passing_options = passing_options.rename(columns={
    "x_start": "receiver_x",
    "y_start": "receiver_y"
})


# ==================================================
# 2️⃣ EXTRACT PLAYER POSSESSION DATA
# ==================================================

possession_events = events[
    events["event_type"] == "player_possession"
].copy()

possession_events = possession_events[
    [
        "match_id",
        "event_id",
        "player_id",
        "player_name",
        "x_start",
        "y_start",
        "player_targeted_id",
        "player_targeted_name",
        "player_targeted_xpass_completion",
        "pass_outcome",
        "pass_distance",
        "pass_angle",
        "pass_direction",
        "pass_range"
    ]
]

# Rename for clarity
possession_events = possession_events.rename(columns={
    "event_id": "possession_event_id",

    "player_id": "player_in_possession_id",
    "player_name": "player_in_possession_name",

    "x_start": "passer_x",
    "y_start": "passer_y",

    "player_targeted_id": "target_receiver_id",
    "player_targeted_xpass_completion": "xpass_from_model"
})

# ==================================================
# 3️⃣ MERGE POSSESSION INFO INTO PASSING OPTIONS
# ==================================================

passing_options = passing_options.merge(
    possession_events,
    left_on=["match_id", "associated_player_possession_event_id"],
    right_on=["match_id", "possession_event_id"],
    how="left"
)


# ==================================================
# 4️⃣ CREATE xPASS + PASS OUTCOME ONLY FOR TARGETED RECEIVER
# ==================================================

passing_options["xpass_completion"] = None
passing_options["actual_pass_outcome"] = None

mask = passing_options["player_id"] == passing_options["target_receiver_id"]

passing_options.loc[mask, "xpass_completion"] = passing_options.loc[
    mask, "xpass_from_model"
]

passing_options.loc[mask, "actual_pass_outcome"] = passing_options.loc[
    mask, "pass_outcome"
]


# ==================================================
# 5️⃣ CLEAN UNNECESSARY COLUMNS
# ==================================================

passing_options = passing_options.drop(columns=[
    "target_receiver_id",
    "xpass_from_model",
    "possession_event_id",
    "pass_outcome"
], errors="ignore")


# ==================================================
# 6️⃣ REORDER COLUMNS (CLEAR STRUCTURE)
# ==================================================

ordered_columns = [

# MATCH CONTEXT
"match_id",
"time_start",
"period",
"team_shortname",

# PASSER (PLAYER IN POSSESSION)
"player_in_possession_id",
"player_in_possession_name",
"passer_x",
"passer_y",

# RECEIVER (PASSING OPTION)
"player_id",
"player_name",
"targeted",
"receiver_x",
"receiver_y",
"xthreat",

# PASS RESULT (ONLY FOR TARGETED)
"xpass_completion",
"actual_pass_outcome",

# ACTUAL PASS CHARACTERISTICS (ONLY TRUE PASS)
"pass_distance",
"pass_angle",
"pass_direction",
"pass_range",

# CONTEXT FEATURES
"channel_start",
"third_start",
"penalty_area_start",
"n_passing_options",
"n_passing_options_ahead",
"n_opponents_ahead_player_in_possession_pass_moment",
"n_opponents_ahead_pass_reception",
"n_opponents_bypassed",
"interplayer_distance",
"interplayer_angle",
"interplayer_direction",
"organised_defense"
]

ordered_columns = [c for c in ordered_columns if c in passing_options.columns]
passing_options = passing_options[ordered_columns]


# ==================================================
# 7️⃣ RESULT
# ==================================================

print("Final dataset shape:", passing_options.shape)

In [ ]:
passing_options.head(10)

In [ ]:
# Data preparation and feature engineering

# ==================================================
# 0️⃣ IMPORTS
# ==================================================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ==================================================
# 1️⃣ TARGET & DATA SPLIT
# ==================================================
target = "xpass_completion"

train_df = passing_options[passing_options[target].notna()].copy()
predict_df = passing_options[passing_options[target].isna()].copy()
train_df[target] = train_df[target].astype(float)

# ==================================================
# 2️⃣ FEATURE ENGINEERING
# ==================================================

# Angle features
train_df["interplayer_angle_sin"] = np.sin(train_df["interplayer_angle"])
train_df["interplayer_angle_cos"] = np.cos(train_df["interplayer_angle"])
predict_df["interplayer_angle_sin"] = np.sin(predict_df["interplayer_angle"])
predict_df["interplayer_angle_cos"] = np.cos(predict_df["interplayer_angle"])

# Differences between passer and receiver
train_df["passer_receiver_x_diff"] = train_df["receiver_x"] - train_df["passer_x"]
train_df["passer_receiver_y_diff"] = train_df["receiver_y"] - train_df["passer_y"]
predict_df["passer_receiver_x_diff"] = predict_df["receiver_x"] - predict_df["passer_x"]
predict_df["passer_receiver_y_diff"] = predict_df["receiver_y"] - train_df["passer_y"]

# Features lists
spatial_features = ["passer_x","passer_y","receiver_x","receiver_y","interplayer_angle"]
tabular_features = [
    "xthreat", "interplayer_distance",
    "n_opponents_ahead_player_in_possession_pass_moment"
]
categorical_features = [
    "interplayer_direction", "channel_start", "third_start",
    "penalty_area_start", "organised_defense"
]
additional_features = ["interplayer_angle_sin", "interplayer_angle_cos", 
                       "passer_receiver_x_diff","passer_receiver_y_diff"]

features_all = spatial_features + tabular_features + categorical_features + additional_features

# Convert categorical to string
for col in categorical_features:
    train_df[col] = train_df[col].astype(str)
    predict_df[col] = predict_df[col].astype(str)

# One-hot encode categorical features for tree models
X = pd.get_dummies(train_df[tabular_features + categorical_features + additional_features])
y = train_df[target]

X_pred = pd.get_dummies(predict_df[tabular_features + categorical_features + additional_features])
X_pred = X_pred.reindex(columns=X.columns, fill_value=0)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=14)

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))

In [ ]:
# Train tree models + neural network

# ==================================================
# 3️⃣ TREE MODELS
# ==================================================
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from sklearn.metrics import mean_squared_error, mean_absolute_error
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import ParameterGrid

# Define models
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=200, max_depth=12, n_jobs=-1, random_state=14),
    "Hist Gradient Boosting": HistGradientBoostingRegressor(max_iter=400, learning_rate=0.05, max_depth=6, random_state=14),
    "XGBoost": xgb.XGBRegressor(n_estimators=500, learning_rate=0.03, max_depth=6,
                                subsample=0.8, colsample_bytree=0.8, random_state=14, n_jobs=-1),
    "LightGBM": lgb.LGBMRegressor(n_estimators=500, learning_rate=0.03, max_depth=6,
                                  subsample=0.8, colsample_bytree=0.8, random_state=14, n_jobs=-1),
    "CatBoost": cb.CatBoostRegressor(iterations=500, learning_rate=0.03, depth=6,
                                     verbose=0, random_state=14)
}

# Train tree models
results = []
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    preds = np.clip(model.predict(X_test), 0, 1)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    results.append((name, rmse, mae))
    print(f"{name} evaluation: RMSE={rmse:.4f}, MAE={mae:.4f}\n{'-'*40}")

results_df = pd.DataFrame(results, columns=["Model","RMSE","MAE"])
best_model_name = results_df.sort_values("RMSE").iloc[0]["Model"]
best_model = models[best_model_name]
print("Best single model:", best_model_name)

# ==================================================
# 4️⃣ NEURAL NETWORK
# ==================================================
# Prepare tabular + spatial tensors
X_tab = pd.get_dummies(train_df[tabular_features + categorical_features + additional_features]).astype(float)
X_tab = X_tab.reindex(columns=X_tab.columns, fill_value=0)
X_tab_t = torch.tensor(X_tab.values, dtype=torch.float32)
X_spatial_t = torch.tensor(train_df[spatial_features].values, dtype=torch.float32)
y_t = torch.tensor(y.values.reshape(-1,1), dtype=torch.float32)

X_tab_tr, X_tab_val, X_sp_tr, X_sp_val, y_tr, y_val = train_test_split(
    X_tab_t, X_spatial_t, y_t, test_size=0.1, random_state=14
)

train_loader = DataLoader(TensorDataset(X_tab_tr, X_sp_tr, y_tr), batch_size=2048, shuffle=True)
val_loader = DataLoader(TensorDataset(X_tab_val, X_sp_val, y_val), batch_size=2048, shuffle=False)

# Neural net class
class SpatialPassNN(nn.Module):
    def __init__(self, tab_dim, spatial_dim, tab_hidden=128, spatial_hidden=64, dropout=0.3):
        super().__init__()
        self.tab_branch = nn.Sequential(
            nn.Linear(tab_dim, tab_hidden), nn.ReLU(),
            nn.BatchNorm1d(tab_hidden), nn.Dropout(dropout),
            nn.Linear(tab_hidden, tab_hidden//2), nn.ReLU()
        )
        self.spatial_branch = nn.Sequential(
            nn.Linear(spatial_dim, spatial_hidden), nn.ReLU(),
            nn.BatchNorm1d(spatial_hidden), nn.Dropout(dropout),
            nn.Linear(spatial_hidden, spatial_hidden//2), nn.ReLU()
        )
        self.combined = nn.Sequential(
            nn.Linear(tab_hidden//2 + spatial_hidden//2, 64), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64,1), nn.Sigmoid()
        )
    def forward(self, x_tab, x_spatial):
        t = self.tab_branch(x_tab)
        s = self.spatial_branch(x_spatial)
        return self.combined(torch.cat([t,s], dim=1))

# Hyperparameter grid search
param_grid = {"tab_hidden":[64,128], "spatial_hidden":[32,64], "dropout":[0.2,0.3], "lr":[0.001,0.0005]}
best_val_rmse = float('inf')
best_params = None
best_model_state = None

for params in ParameterGrid(param_grid):
    model = SpatialPassNN(tab_dim=X_tab_t.shape[1], spatial_dim=X_spatial_t.shape[1],
                          tab_hidden=params["tab_hidden"], spatial_hidden=params["spatial_hidden"],
                          dropout=params["dropout"])
    optimizer = optim.Adam(model.parameters(), lr=params["lr"])
    criterion = nn.MSELoss()
    patience = 5
    patience_counter = 0
    for epoch in range(20):
        model.train()
        for xb_tab, xb_sp, yb in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(xb_tab, xb_sp), yb)
            loss.backward()
            optimizer.step()
        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for xb_tab, xb_sp, yb in val_loader:
                val_loss += criterion(model(xb_tab, xb_sp), yb).item() * xb_tab.size(0)
        val_rmse = np.sqrt(val_loss / len(val_loader.dataset))
        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_params = params
            best_model_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

print("Best NN params:", best_params, "Val RMSE:", best_val_rmse)

In [ ]:
# ==================================================
# 6️⃣ PREDICT USING BEST NN MODEL
# ==================================================

# Recreate best NN
best_nn = SpatialPassNN(
    tab_dim=X_tab_t.shape[1],
    spatial_dim=X_spatial_t.shape[1],
    tab_hidden=best_params["tab_hidden"],
    spatial_hidden=best_params["spatial_hidden"],
    dropout=best_params["dropout"]
)
best_nn.load_state_dict(best_model_state)
best_nn.eval()

# Prepare prediction tensors
X_pred_tab = pd.get_dummies(predict_df[tabular_features + categorical_features]).astype(float)
X_pred_tab = X_pred_tab.reindex(columns=X_tab_t.shape[1]*[0], fill_value=0)  # align columns if needed
X_pred_tab_t = torch.tensor(X_pred_tab.values, dtype=torch.float32)
X_pred_spatial_t = torch.tensor(predict_df[spatial_features].values, dtype=torch.float32)

# Predict
with torch.no_grad():
    preds = best_nn(X_pred_tab_t, X_pred_spatial_t).numpy().flatten()
preds = np.clip(preds, 0, 1)

# Add predictions to your DataFrame
predict_df["predicted_xpass_nn"] = preds
passing_options["predicted_xpass_nn"] = None
passing_options.loc[predict_df.index, "predicted_xpass_nn"] = preds

print("NN predictions added.")
print(passing_options[["xpass_completion","predicted_xpass_nn"]].head())
print("Prediction range:", passing_options["predicted_xpass_nn"].min(), passing_options["predicted_xpass_nn"].max())

In [ ]:
# ==================================================
# 6️⃣ PREDICT USING BEST TREE MODEL (XGBoost)
# ==================================================

# Prepare prediction features (tabular + categorical + spatial + engineered)
X_pred = pd.get_dummies(predict_df[features]).astype(float)
# Align columns with training
X_pred = X_pred.reindex(columns=model_columns, fill_value=0)

# Predict with XGBoost
preds = best_model.predict(X_pred)
preds = np.clip(preds, 0, 1)

# Add predictions to DataFrame
predict_df["predicted_xpass"] = preds
passing_options["predicted_xpass"] = None
passing_options.loc[predict_df.index, "predicted_xpass"] = preds

print("XGBoost predictions added.")
print(passing_options[["xpass_completion", "predicted_xpass"]].head())
print("Prediction range:", passing_options["predicted_xpass"].min(), passing_options["predicted_xpass"].max())

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Get XGBoost feature importance
booster = best_model.get_booster()  # best_model is your trained XGBoost
score_dict = booster.get_score(importance_type='gain')
features = booster.feature_names
gains = [score_dict.get(f, 0) for f in features]

# Create DataFrame
importance_df = pd.DataFrame({"feature": features, "gain": gains})
importance_df["gain"] = importance_df["gain"].round(3)
importance_df = importance_df.sort_values("gain", ascending=True)

# Plot
plt.figure(figsize=(10,6))
plt.barh(importance_df["feature"], importance_df["gain"], color='skyblue')
plt.xlabel("Gain (importance)")
plt.title("XGBoost Feature Importance")
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
from pathlib import Path

# --------------------------------------------------
# PANDAS DISPLAY SETTINGS
# --------------------------------------------------
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.expand_frame_repr", False)

# ==================================================
# 1. LOAD ALL MATCH EVENT FILES
# ==================================================
folder = Path(r"Datasets\SkillCorner Premier League 24-25 data\dynamic_events_pl_24\dynamic")
dfs = [pd.read_parquet(file) for file in folder.glob("*.parquet")]
events = pd.concat(dfs, ignore_index=True)

print("Total events loaded:", len(events))
print("Unique matches:", events["match_id"].nunique())

# ==================================================
# 2. FILTER PASSING OPTIONS (includes all options and actual passes)
# ==================================================
passing_options = events[events["event_type"] == "passing_option"].copy()
print("\nPassing option rows:", len(passing_options))

# ==================================================
# 3. KEEP IMPORTANT COLUMNS & CLEAN
# ==================================================
passing_options = passing_options[
    [
        "match_id",
        "associated_player_possession_event_id",
        "player_id",
        "player_name",
        "passing_option_score",
        "xthreat",
        "targeted"
    ]
].dropna(subset=["passing_option_score"])

# Convert numeric columns
passing_options["xthreat"] = pd.to_numeric(passing_options["xthreat"], errors="coerce").fillna(0)

print("\nRows after cleaning:", len(passing_options))

# ==================================================
# 4. COUNT OPTIONS PER DECISION
# ==================================================
option_counts = (
    passing_options
    .groupby(["match_id", "associated_player_possession_event_id"])
    .size()
    .reset_index(name="n_options")
)
print("\nDecision moments:", len(option_counts))

# ==================================================
# 5. REMOVE TRIVIAL DECISIONS (only keep decisions with 2+ options)
# ==================================================
option_counts = option_counts[option_counts["n_options"] >= 2]
passing_options = passing_options.merge(
    option_counts[["match_id", "associated_player_possession_event_id"]],
    on=["match_id", "associated_player_possession_event_id"]
)
print("\nPassing options after filtering trivial decisions:", len(passing_options))

# ==================================================
# 6. PASS VALUE (simplified)
# ==================================================
passing_options["pass_value"] = passing_options["xthreat"]

# ==================================================
# 7. BEST OPTION PER DECISION
# ==================================================
best_option = (
    passing_options
    .groupby(["match_id", "associated_player_possession_event_id"])
    .agg(
        best_option_score=("passing_option_score", "max"),
        best_xThreat_available=("xthreat", "max"),
        best_pass_value=("pass_value", "max"),
        avg_option_quality=("passing_option_score", "mean"),
        avg_option_xThreat=("xthreat", "mean")
    )
    .reset_index()
)

# ==================================================
# 8. IDENTIFY CHOSEN PASSES
# ==================================================
chosen = passing_options[passing_options["targeted"] == True].copy()
chosen = chosen[
    [
        "match_id",
        "associated_player_possession_event_id",
        "player_id",
        "player_name",
        "passing_option_score",
        "xthreat",
        "pass_value"
    ]
].rename(columns={
    "passing_option_score": "chosen_option_score",
    "xthreat": "chosen_xThreat",
    "pass_value": "chosen_pass_value"
})

# ==================================================
# 9. MERGE DECISION DATA
# ==================================================
decisions = chosen.merge(
    best_option,
    on=["match_id", "associated_player_possession_event_id"]
)

# ==================================================
# 10. COMPUTE DECISION METRICS
# ==================================================
decisions["decision_quality"] = decisions["chosen_option_score"] / decisions["best_option_score"]
decisions["value_decision_quality"] = decisions["chosen_pass_value"] / decisions["best_pass_value"]
decisions["creativity"] = decisions["chosen_xThreat"] / decisions["best_xThreat_available"]
decisions["risk_taken"] = decisions["best_option_score"] - decisions["chosen_option_score"]
decisions["xThreat_gain"] = decisions["chosen_xThreat"] - decisions["best_xThreat_available"]

# ==================================================
# 11. AGGREGATE PLAYER METRICS
# ==================================================
players = (
    decisions
    .groupby(["player_id", "player_name"])
    .agg(
        passes=("decision_quality", "count"),
        decision_quality=("decision_quality", "mean"),
        value_decision_quality=("value_decision_quality", "mean"),
        creativity=("creativity", "mean"),
        avg_xThreat_created=("chosen_xThreat", "mean"),
        avg_xThreat_gain=("xThreat_gain", "mean"),
        total_xThreat_gain=("xThreat_gain", "sum"),
        avg_option_quality=("avg_option_quality", "mean"),
        risk_taken=("risk_taken", "sum")  # sum for per90
    )
    .reset_index()
)
players = players[players["passes"] > 200]
print("Players with 200+ decisions:", len(players))

# ==================================================
# 12. LOAD PLAYER POSITION + MINUTES
# ==================================================
players_match = pd.read_parquet(r"Datasets/SkillCorner Premier League 24-25 data/players_match.parquet")
players_match["player_id"] = players_match["id"]
players_match["player_name"] = players_match["short_name"]
players_match["position"] = players_match["player_role_acronym"]
players_match["position_group"] = players_match["player_role_position_group"]

position_counts = (
    players_match
    .groupby(["player_id","player_name","team_id","position","position_group"])
    .size()
    .reset_index(name="matches_at_position")
)

# ==================================================
# 12a. HANDLE SUBSTITUTES AND SECOND MAIN POSITION
# ==================================================
position_counts_sorted = position_counts.sort_values(["player_id","matches_at_position"], ascending=[True, False])

def get_main_and_group(df):
    main_pos = df.iloc[0]["position"]
    sub_flag = 0
    final_pos = main_pos
    final_group = df.iloc[0]["position_group"]
    if main_pos == "SUB":
        sub_flag = 1
        if len(df) > 1:
            final_pos = df.iloc[1]["position"]
            final_group = df.iloc[1]["position_group"]
        else:
            final_pos = "SUB"
            final_group = "SUB"
    # Special case for GK
    if final_pos == "GK":
        final_group = "Goalkeeper"
    return pd.Series({"main_position_adjusted": final_pos, "SUB": sub_flag, "position_group_adjusted": final_group})

main_positions_adjusted = position_counts_sorted.groupby("player_id").apply(get_main_and_group).reset_index()

# ==================================================
# Player info with minutes
# ==================================================
player_main_position = position_counts.sort_values("matches_at_position", ascending=False).drop_duplicates("player_id")
player_main_position = player_main_position[["player_id","team_id","position","position_group"]].rename(columns={"position":"main_position"})

minutes_df = (
    players_match
    .groupby("player_id")
    .agg(
        minutes_played_total=("playing_time_total_minutes_played","sum"),
        minutes_regular_time=("playing_time_total_minutes_played_regular_time","sum")
    )
    .reset_index()
)
minutes_df["per90_factor"] = minutes_df["minutes_played_total"] / 90

player_info = player_main_position.merge(minutes_df, on="player_id", how="left")
player_info = player_info.merge(main_positions_adjusted, on="player_id", how="left")
player_info["main_position"] = player_info["main_position_adjusted"]
player_info["position_group"] = player_info["position_group_adjusted"]
player_info.drop(columns=["main_position_adjusted","position_group_adjusted"], inplace=True)

# ==================================================
# 13. LOAD TEAM NAMES
# ==================================================
matches = pd.read_parquet(r"Datasets/SkillCorner Premier League 24-25 data/matches_clean.parquet")
team_lookup = pd.concat([
    matches[["home_team_id","home_team_name"]].rename(columns={"home_team_id":"team_id","home_team_name":"team_name"}),
    matches[["away_team_id","away_team_name"]].rename(columns={"away_team_id":"team_id","away_team_name":"team_name"})
]).drop_duplicates()

player_info = player_info.merge(team_lookup, on="team_id", how="left")

# ==================================================
# 14. MERGE PLAYER METRICS WITH INFO
# ==================================================
players = players.merge(player_info, on="player_id", how="left")

# ==================================================
# 15. CALCULATE PER 90 METRICS (only sums/counts)
# ==================================================
players["passes_per90"] = players["passes"] / players["per90_factor"]
players["total_xThreat_gain_per90"] = players["total_xThreat_gain"] / players["per90_factor"]
players["risk_taken_per90"] = players["risk_taken"] / players["per90_factor"]

# ==================================================
# 16. FINAL COLUMN ORDER
# ==================================================
columns_order = [
    "player_id","player_name","team_id","team_name",
    "main_position","SUB","position_group",
    "passes","passes_per90",
    "decision_quality","value_decision_quality","creativity",
    "avg_xThreat_created","avg_xThreat_gain",
    "total_xThreat_gain","total_xThreat_gain_per90",
    "avg_option_quality","risk_taken","risk_taken_per90",
    "minutes_played_total","minutes_regular_time","per90_factor"
]
players = players[columns_order]

In [ ]:
# Top decision makers:
players.sort_values("decision_quality", ascending=False).head(10)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set seaborn style
sns.set(style="whitegrid", font_scale=1.2)

# ==================================================
# 1. Scatterplot: decision_quality vs avg_option_quality
# ==================================================
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=players,
    x="avg_option_quality",
    y="decision_quality",
    hue="position_group",
    palette="tab10",
    alpha=0.8,
    s=80
)
plt.title("Decision Quality vs Average Option Quality")
plt.xlabel("Average Option Quality (Passing Options Score)")
plt.ylabel("Decision Quality")
plt.legend(title="Position Group", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# ==================================================
# 2. Scatterplot: decision_quality vs risk_taken_per90
# ==================================================
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=players,
    x="risk_taken_per90",
    y="decision_quality",
    hue="position_group",
    palette="tab10",
    alpha=0.8,
    s=80
)
plt.title("Decision Quality vs Risk Taken per 90")
plt.xlabel("Risk Taken per 90 (Score Difference from Best Option)")
plt.ylabel("Decision Quality")
plt.legend(title="Position Group", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()